<a href="https://colab.research.google.com/github/Chathuranga630/ML-Workshop_SLT/blob/main/UAV_Fleet_Simulation_V1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
"""
Basic UAV fleet coordination simulation.
Python 3.10+; requires only the standard library.

Models:
- 8 UAVs and 12 tasks by default
- Energy-aware greedy assignment under time and battery constraints
- Attack modes: false battery reports, spoofed progress messages, task non-completion
- Optional defense: status verification, reliability scoring, task reassignment
- Repeated trials and CSV/JSON summary output

This is a simplified research prototype, not a flight-ready controller.
"""

from __future__ import annotations
import argparse
import csv
import json
import random
import sys
import statistics
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


@dataclass
class UAV:
    uid: int
    true_battery: float
    speed: float
    energy_per_distance: float
    is_malicious: bool = False
    attack: str = "none"
    reliability: float = 1.0
    detected: bool = False
    available_at: int = 0
    energy_used: float = 0.0


@dataclass
class Task:
    tid: int
    location: float
    workload: int
    deadline: int
    assigned_to: Optional[int] = None
    completed: bool = False
    start_time: Optional[int] = None
    finish_time: Optional[int] = None
    attempts: int = 0


def reported_battery(uav: UAV, rng: random.Random) -> float:
    """A compromised UAV exaggerates its battery level."""
    if uav.attack == "false_battery":
        return min(100.0, uav.true_battery + rng.uniform(25, 45))
    return uav.true_battery


def estimated_task_cost(uav: UAV, task: Task, now: int) -> tuple[float, int]:
    """Toy 1-D travel model; a busy UAV can start after its current assignment."""
    travel_distance = abs(task.location)
    travel_time = max(1, int((travel_distance / max(uav.speed, 0.1)) + 0.999))
    total_time = travel_time + task.workload
    start_time = max(now, uav.available_at)
    energy = travel_distance * uav.energy_per_distance + task.workload * 1.5
    return energy, start_time + total_time


def create_world(seed: int, n_uavs: int, n_tasks: int, attack_mode: str,
                 attack_fraction: float) -> tuple[list[UAV], list[Task]]:
    rng = random.Random(seed)
    uavs = []
    for i in range(n_uavs):
        uavs.append(UAV(
            uid=i,
            true_battery=rng.uniform(65, 100),
            speed=rng.uniform(1.5, 3.0),
            energy_per_distance=rng.uniform(0.7, 1.3),
        ))

    n_bad = round(n_uavs * attack_fraction) if attack_mode != "none" else 0
    compromised = set(rng.sample(range(n_uavs), min(n_bad, n_uavs)))
    for i in compromised:
        uavs[i].is_malicious = True
        uavs[i].attack = attack_mode

    tasks = []
    for j in range(n_tasks):
        loc = rng.uniform(3, 20)
        workload = rng.randint(2, 6)
        # The deadline is challenging but not impossible for most tasks.
        deadline = rng.randint(18, 40)
        tasks.append(Task(j, loc, workload, deadline))
    return uavs, tasks


def allocate(uavs: list[UAV], task: Task, now: int, rng: random.Random,
             defended: bool, blacklist: set[int]) -> Optional[UAV]:
    candidates = []
    for u in uavs:
        if u.uid in blacklist:
            continue
        energy, finish = estimated_task_cost(u, task, now)
        batt = reported_battery(u, rng) if not defended else u.true_battery
        if batt < energy + 8:
            continue
        if finish > task.deadline:
            continue
        # Prefer low energy, then shorter finish time. A small reliability penalty
        # reduces trust in repeatedly inconsistent agents.
        score = energy + max(0.0, 1.0 - u.reliability) * 12
        candidates.append((score, finish, u.uid, u))
    if not candidates:
        return None
    return min(candidates, key=lambda x: (x[0], x[1], x[2]))[3]


def run_trial(seed: int, n_uavs: int = 8, n_tasks: int = 12,
              attack_mode: str = "none", attack_fraction: float = 0.25,
              defended: bool = False) -> dict:
    rng = random.Random(seed + 100_003)
    uavs, tasks = create_world(seed, n_uavs, n_tasks, attack_mode, attack_fraction)
    blacklist: set[int] = set()
    now = 0
    total_energy = 0.0
    detections = 0
    reassigned = 0
    attempts = 0
    false_success_claims = 0
    successful_tasks = 0
    on_time_tasks = 0

    # Schedule tasks in earliest-deadline-first order.
    for task in sorted(tasks, key=lambda t: (t.deadline, t.tid)):
        # Each task has its own deadline measured from mission start (time zero).
        # UAV availability persists across tasks, allowing parallel fleet scheduling.
        now = 0
        task.assigned_to = None
        task.completed = False
        task.start_time = None
        task.finish_time = None
        tries_for_task = 0

        while tries_for_task < 3 and not task.completed:
            chosen = allocate(uavs, task, now, rng, defended, blacklist)
            if chosen is None:
                break

            tries_for_task += 1
            attempts += 1
            task.attempts += 1
            energy, predicted_finish = estimated_task_cost(chosen, task, now)

            # True battery always pays the physical energy cost in the model.
            # A battery-spoofing UAV may have looked feasible under baseline policy.
            actual_energy = energy
            chosen.true_battery -= actual_energy
            chosen.energy_used += actual_energy
            total_energy += actual_energy
            chosen.available_at = predicted_finish
            task.assigned_to = chosen.uid
            if task.start_time is None:
                task.start_time = now

            # Failure-to-complete attack: the UAV consumes energy and does not finish.
            fail = chosen.attack in ("task_failure", "spoofed_messages")
            # Spoofed-message attackers claim success despite not completing the task.
            spoofed_success_claim = chosen.attack == "spoofed_messages"

            if fail:
                if defended:
                    chosen.reliability = max(0.0, chosen.reliability - 0.55)
                    if not chosen.detected:
                        chosen.detected = True
                        detections += 1
                    blacklist.add(chosen.uid)
                    reassigned += 1
                    # Retry the task from the current simulation time with another UAV.
                    now = max(now, chosen.available_at)
                    continue
                else:
                    if spoofed_success_claim:
                        # Baseline trusts the forged completion message. Count the false
                        # claim separately, but do not mark the task truly completed.
                        false_success_claims += 1
                        task.finish_time = predicted_finish
                        task.assigned_to = chosen.uid
                    now = max(now, chosen.available_at)
                    break

            # The defended coordinator validates completion against expected progress.
            # This toy model assumes non-failing UAVs complete the task.
            task.completed = True
            task.finish_time = predicted_finish
            successful_tasks += 1
            if predicted_finish <= task.deadline:
                on_time_tasks += 1
            now = max(now, min(chosen.available_at, task.deadline))

        if not task.completed and defended:
            # If no feasible UAV can complete by the deadline, record failure.
            pass

    # Detect false-battery behavior using report-vs-verified discrepancy.
    if defended:
        for u in uavs:
            if u.attack == "false_battery" and not u.detected:
                # Periodic verification reveals an implausibly large discrepancy.
                if reported_battery(u, rng) - u.true_battery > 20:
                    u.detected = True
                    u.reliability = max(0.0, u.reliability - 0.5)
                    blacklist.add(u.uid)
                    detections += 1

    completed_truth = sum(1 for t in tasks if t.completed)
    on_time_truth = sum(1 for t in tasks if t.completed and
                        t.finish_time is not None and t.finish_time <= t.deadline)
    malicious_count = sum(1 for u in uavs if u.is_malicious)
    detected_malicious = sum(1 for u in uavs if u.is_malicious and u.detected)

    return {
        "seed": seed,
        "attack_mode": attack_mode,
        "defense_enabled": defended,
        "uavs": n_uavs,
        "tasks": n_tasks,
        "malicious_uavs": malicious_count,
        "completed_tasks": completed_truth,
        "completion_rate": completed_truth / n_tasks,
        "on_time_tasks": on_time_truth,
        "on_time_rate": on_time_truth / n_tasks,
        "total_energy": round(total_energy, 2),
        "average_energy_per_completed_task": round(total_energy / max(1, completed_truth), 2),
        "detections": detections,
        "false_success_claims": false_success_claims,
        "malicious_detection_rate": (detected_malicious / malicious_count) if malicious_count else 0.0,
        "reassignments": reassigned,
        "assignment_attempts": attempts,
    }


def run_experiment(trials: int, n_uavs: int, n_tasks: int, attack_fraction: float):
    scenarios = [
        ("none", False, "No attack / baseline"),
        ("false_battery", False, "False battery / baseline"),
        ("spoofed_messages", False, "Spoofed messages / baseline"),
        ("task_failure", False, "Task failure / baseline"),
        ("false_battery", True, "False battery / defended"),
        ("spoofed_messages", True, "Spoofed messages / defended"),
        ("task_failure", True, "Task failure / defended"),
    ]
    all_rows = []
    summary = []
    for mode, defended, label in scenarios:
        rows = [
            run_trial(seed=1000 + i, n_uavs=n_uavs, n_tasks=n_tasks,
                      attack_mode=mode, attack_fraction=attack_fraction,
                      defended=defended)
            for i in range(trials)
        ]
        all_rows.extend(rows)
        summary.append({
            "scenario": label,
            "mean_completion_rate": round(statistics.mean(r["completion_rate"] for r in rows), 3),
            "mean_on_time_rate": round(statistics.mean(r["on_time_rate"] for r in rows), 3),
            "mean_total_energy": round(statistics.mean(r["total_energy"] for r in rows), 2),
            "mean_energy_per_completed_task": round(statistics.mean(
                r["average_energy_per_completed_task"] for r in rows), 2),
            "mean_malicious_detection_rate": round(statistics.mean(
                r["malicious_detection_rate"] for r in rows), 3),
            "mean_reassignments": round(statistics.mean(r["reassignments"] for r in rows), 2),
            "mean_false_success_claims": round(statistics.mean(r["false_success_claims"] for r in rows), 2),
        })
    return all_rows, summary


def main(args_list: Optional[list[str]] = None):
    parser = argparse.ArgumentParser(description="Basic UAV fleet coordination and attack simulation")
    parser.add_argument("--uavs", type=int, default=8)
    parser.add_argument("--tasks", type=int, default=12)
    parser.add_argument("--trials", type=int, default=50)
    parser.add_argument("--attack-fraction", type=float, default=0.25)
    parser.add_argument("--out", type=Path, default=Path("uav_results"))

    # If running in a notebook, parse empty/custom list to prevent parsing Jupyter argv
    if args_list is None and hasattr(sys, 'ps1'):
        args = parser.parse_args([])
    else:
        args = parser.parse_args(args_list)

    if args.uavs < 1 or args.tasks < 1 or args.trials < 1:
        parser.error("--uavs, --tasks, and --trials must be >= 1")
    if not 0 <= args.attack_fraction <= 1:
        parser.error("--attack-fraction must be between 0 and 1")

    args.out.mkdir(parents=True, exist_ok=True)
    rows, summary = run_experiment(args.trials, args.uavs, args.tasks, args.attack_fraction)

    with (args.out / "trial_results.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    with (args.out / "scenario_summary.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary[0].keys()))
        writer.writeheader()
        writer.writerows(summary)

    (args.out / "scenario_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"Trials per scenario: {args.trials}")
    print(f"UAVs: {args.uavs}; tasks: {args.tasks}; attack fraction: {args.attack_fraction:.0%}")
    print("-" * 105)
    print(f"{'Scenario':32} {'Complete':>9} {'On time':>9} {'Energy':>10} {'Energy/task':>12} {'Detect':>8} {'Reassign':>9}")
    for s in summary:
        print(f"{s['scenario']:32} {s['mean_completion_rate']:9.1%} {s['mean_on_time_rate']:9.1%} "
              f"{s['mean_total_energy']:10.1f} {s['mean_energy_per_completed_task']:12.1f} "
              f"{s['mean_malicious_detection_rate']:8.1%} {s['mean_reassignments']:9.2f}")
    print(f"\nCSV and JSON results saved in: {args.out.resolve()}")


if __name__ == "__main__":
    main()

Trials per scenario: 50
UAVs: 8; tasks: 12; attack fraction: 25%
---------------------------------------------------------------------------------------------------------
Scenario                          Complete   On time     Energy  Energy/task   Detect  Reassign
No attack / baseline                100.0%    100.0%      187.4         15.6     0.0%      0.00
False battery / baseline            100.0%    100.0%      189.7         15.8     0.0%      0.00
Spoofed messages / baseline          74.7%     74.7%      189.7         23.4     0.0%      0.00
Task failure / baseline              74.7%     74.7%      189.7         23.4     0.0%      0.00
False battery / defended            100.0%    100.0%      189.7         15.8    71.0%      0.00
Spoofed messages / defended          97.0%     97.0%      210.3         18.1    59.0%      1.18
Task failure / defended              97.0%     97.0%      210.3         18.1    59.0%      1.18

CSV and JSON results saved in: /content/uav_results
